# TLAQ — Temporal Knowledge Graph Approximate Query

End-to-end experiment on QALD-6 / QALD-7 using DBpedia as TKG.

**Runtime**: GPU, Python 3.10+  
**Expected time**: ~30-40 minutes (full DBpedia scan + training)

## Step 0 — Clone repo & install dependencies

In [ ]:
import os

# Clone the TLAQ repo
if not os.path.exists('/kaggle/working/TLAQ'):
    !git clone https://github.com/LjxAbi/TLAQ.git /kaggle/working/TLAQ
else:
    !git -C /kaggle/working/TLAQ pull

os.chdir('/kaggle/working/TLAQ')
!pwd

In [ ]:
# Kaggle already has torch; only need to verify version
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 1 — Set data paths

After uploading the datasets (see README below), update the paths here.

In [ ]:
import os

# ── Adjust these paths to match your Kaggle dataset names ──────────────────
DBPEDIA_OBJECTS  = '/kaggle/input/dbpedia-tlaq/mappingbased-objects_lang=en.ttl.bz2'
DBPEDIA_LITERALS = '/kaggle/input/dbpedia-tlaq/mappingbased-literals_lang=en.ttl.bz2'
QALD7_TRAIN      = '/kaggle/input/qald-tlaq/qald-7-train-multilingual.json'
QALD7_TEST       = '/kaggle/input/qald-tlaq/qald-7-test-multilingual.json'
QALD6_TRAIN      = '/kaggle/input/qald-tlaq/qald-6-train-multilingual.json'
QALD6_TEST       = '/kaggle/input/qald-tlaq/qald-6-test-multilingual.json'
# ────────────────────────────────────────────────────────────────────────────

for path in [DBPEDIA_OBJECTS, DBPEDIA_LITERALS, QALD7_TRAIN, QALD6_TRAIN]:
    status = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f'{status}  {path}')

## Step 2 — Run experiment on QALD-7

In [ ]:
!python scripts/run_experiment.py \
    --qald {QALD7_TRAIN} {QALD7_TEST} \
    --dbpedia-objects  {DBPEDIA_OBJECTS} \
    --dbpedia-literals {DBPEDIA_LITERALS} \
    --background-limit 300000 \
    --embed-dim 128 \
    --epochs 100 \
    --lr 1e-3 \
    --batch-size 128 \
    --patience 10 \
    --top-k 200 \
    --device cuda \
    --run-name qald7_full

## Step 3 — Run experiment on QALD-6

In [ ]:
!python scripts/run_experiment.py \
    --qald {QALD6_TRAIN} {QALD6_TEST} \
    --dbpedia-objects  {DBPEDIA_OBJECTS} \
    --dbpedia-literals {DBPEDIA_LITERALS} \
    --background-limit 300000 \
    --embed-dim 128 \
    --epochs 100 \
    --lr 1e-3 \
    --batch-size 128 \
    --patience 10 \
    --top-k 200 \
    --device cuda \
    --run-name qald6_full

## Step 4 — View results

In [ ]:
import json

for run in ['qald7_full', 'qald6_full']:
    metrics_path = f'results/{run}/metrics.json'
    if not os.path.exists(metrics_path):
        print(f'{run}: not yet completed')
        continue
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f'\n=== {run} ===')
    for dataset, scores in metrics.items():
        print(f'  [{dataset}]')
        for k in sorted(scores, key=lambda s: (int(s.split("@")[1]), s.split("@")[0])):
            print(f'    {k} = {scores[k]:.4f}')

## Step 5 — Plot loss curves (optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, run in zip(axes, ['qald7_full', 'qald6_full']):
    history_path = f'results/{run}/loss_history.json'
    if not os.path.exists(history_path):
        ax.set_title(f'{run} — not completed')
        continue
    with open(history_path) as f:
        history = json.load(f)
    if history.get('gcn'):
        ax.plot(history['gcn'],  label='GCN')
    if history.get('hyte'):
        ax.plot(history['hyte'], label='HyTE')
    ax.set_title(f'{run} — Loss Curve')
    ax.set_xlabel('Epoch (×log_every)')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/loss_curves.png', dpi=150)
plt.show()
print('Saved to results/loss_curves.png')

---
## Data Upload Guide

Before running this notebook, upload two datasets to Kaggle:

### Dataset 1: `dbpedia-tlaq`
Files to upload:
- `mappingbased-objects_lang=en.ttl.bz2`  (~188 MB)
- `mappingbased-literals_lang=en.ttl.bz2` (~151 MB)

### Dataset 2: `qald-tlaq`
Files to upload:
- `qald-7-train-multilingual.json`
- `qald-7-test-multilingual.json`
- `qald-6-train-multilingual.json`
- `qald-6-test-multilingual.json`

**How to upload:**
1. Go to https://www.kaggle.com/datasets
2. Click **New Dataset**
3. Drag and drop files, name the dataset
4. In this notebook, click **Add Data** (right panel) and search your dataset name
5. Update the paths in Step 1 if needed